## 1. Overview: Office Products Search + Review Summarization

In this notebook we build an end-to-end search engine for Amazon Office Products:

- Aggregate review-level data to the product (parent_asin) level
- Join with metadata (title, categories, average rating, price, etc.)
- Perform Exploratory Data Analysis (EDA) to understand:
  - How many reviews each product has
  - How long reviews are on average
  - How ratings and helpful votes behave
- Build a TF-IDF based search index using a combined text field
- Implement a lightweight extractive summarization for reviews
- Expose the system through a simple Flask dashboard
- Evaluate retrieval quality with Hit@k, MRR@k and nDCG@k

Along the way we explicitly address:

- Why the summarization sometimes looked wrong, and how we improve it
- Why TF-IDF + cosine similarity is a reasonable first-step
- How a future embedding-based index could further improve semantic search


In [ ]:
# Imports and configuration

import os
import json
import pickle
from collections import defaultdict
from typing import Dict, List, Any, Optional

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
import joblib
import re
import matplotlib.pyplot as plt


### 1.1 Paths, hyper-parameters and basic helpers

We configure paths to the Amazon Office Products files and define:

- How many reviews we read in total (for compute control)
- How many reviews per product we keep as text samples for summarization
- Parameters that control how many sentences go into the summary
- A simple sentence splitter based on punctuation


In [ ]:
# Raw data paths
META_PATH = "meta_Office_Products.jsonl"
REVIEW_PATH = "Office_Products.jsonl"

# Index output directory and artifact paths
BASE_DIR = "./office_products_index"
os.makedirs(BASE_DIR, exist_ok=True)

PRODUCT_CSV_PATH = os.path.join(BASE_DIR, "products_index.csv")
TFIDF_VECTORIZER_PATH = os.path.join(BASE_DIR, "tfidf_vectorizer.joblib")
TFIDF_MATRIX_PATH = os.path.join(BASE_DIR, "product_tfidf.npz")
REVIEWS_SAMPLE_PKL = os.path.join(BASE_DIR, "reviews_sample.pkl")

# Controls to limit processing cost
MAX_REVIEWS_TO_READ = 300_000          # Max reviews to read from the big review file
MAX_REVIEWS_PER_PRODUCT_FOR_SAMPLE = 20  # Max reviews per product kept as text samples

# Parameters for summarization
MAX_SENTENCES_IN_SUMMARY = 3
MAX_SENTENCES_CONSIDERED = 80
MIN_REVIEWS_FOR_SUMMARY = 3          # require at least 3 reviews to trust a summary
MIN_AVG_TEXT_LEN_FOR_SUMMARY = 80    # require sufficient text length

# Simple sentence splitter regex
_SENT_SPLIT_REGEX = re.compile(r"(?<=[.!?])\s+")


def split_into_sentences(text: str) -> List[str]:
    if not text:
        return []
    return _SENT_SPLIT_REGEX.split(text)


def clean_review_text(text: str) -> str:
    """
    Basic text cleaning for reviews:
    - Remove simple HTML tags such as <br/>
    - Collapse repeated whitespace
    """
    if not isinstance(text, str):
        return ""
    no_html = re.sub(r"<[^>]+>", " ", text)
    return re.sub(r"\s+", " ", no_html).strip()


## 2. Aggregate reviews to product level

We first aggregate review data at the `parent_asin` level:

- `n_reviews`: how many reviews a product has
- `avg_rating_reviews`: average rating derived from review records
- `avg_text_len`: average review text length (characters)
- `total_helpful_vote`: sum of helpful votes

At the same time we keep up to `MAX_REVIEWS_PER_PRODUCT_FOR_SAMPLE` cleaned review
texts per product, which will be used later for summarization.


In [ ]:
# Aggregate reviews per parent_asin

print("=" * 80)
print("Checking file paths")
print("=" * 80)
print("META_PATH exists:", os.path.exists(META_PATH))
print("REVIEW_PATH exists:", os.path.exists(REVIEW_PATH))

if not os.path.exists(META_PATH):
    raise FileNotFoundError(f"META file not found: {META_PATH}")
if not os.path.exists(REVIEW_PATH):
    raise FileNotFoundError(f"REVIEW file not found: {REVIEW_PATH}")

print()
print("=" * 80)
print("[STEP 1] Aggregating reviews per parent_asin")
print("=" * 80)

# Aggregation containers
review_count: Dict[str, int] = defaultdict(int)
rating_sum: Dict[str, float] = defaultdict(float)
text_len_sum: Dict[str, float] = defaultdict(float)
helpful_sum: Dict[str, int] = defaultdict(int)

# Text samples for summarization
reviews_sample: Dict[str, List[str]] = defaultdict(list)

total_reviews_processed = 0

with open(REVIEW_PATH, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        total_reviews_processed += 1
        try:
            rec = json.loads(line)
        except json.JSONDecodeError:
            continue

        parent_asin = rec.get("parent_asin")
        text_raw = rec.get("text", "")
        rating = rec.get("rating", None)
        helpful = rec.get("helpful_vote", 0)

        if parent_asin is None:
            continue

        parent_asin = str(parent_asin)
        review_count[parent_asin] += 1

        # Rating aggregation
        if rating is not None:
            try:
                rating_sum[parent_asin] += float(rating)
            except (ValueError, TypeError):
                pass

        # Text length aggregation
        cleaned_text = clean_review_text(text_raw)
        text_len = len(cleaned_text)
        text_len_sum[parent_asin] += text_len

        # Helpful votes aggregation
        try:
            helpful_sum[parent_asin] += int(helpful)
        except (ValueError, TypeError):
            pass

        # Keep a small number of cleaned review texts per product for summarization
        if text_len > 0 and len(reviews_sample[parent_asin]) < MAX_REVIEWS_PER_PRODUCT_FOR_SAMPLE:
            reviews_sample[parent_asin].append(cleaned_text)

        if total_reviews_processed % 50_000 == 0:
            print(f"Processed {total_reviews_processed} reviews...")

        if total_reviews_processed >= MAX_REVIEWS_TO_READ:
            print(f"Reached MAX_REVIEWS_TO_READ={MAX_REVIEWS_TO_READ}, stopping.")
            break

print(f"Total reviews processed: {total_reviews_processed}")
print(f"Unique parent_asin with reviews: {len(review_count)}")

# Build aggregated DataFrame
agg_records: List[Dict[str, Any]] = []
for pa, n in review_count.items():
    if n <= 0:
        continue
    avg_rating_reviews = rating_sum[pa] / n if n > 0 else None
    avg_text_len = text_len_sum[pa] / n if n > 0 else 0.0
    agg_records.append(
        {
            "parent_asin": pa,
            "n_reviews": n,
            "avg_rating_reviews": avg_rating_reviews,
            "avg_text_len": avg_text_len,
            "total_helpful_vote": helpful_sum[pa],
        }
    )

reviews_agg_df = pd.DataFrame(agg_records)
print("Aggregated stats for", len(reviews_agg_df), "products (parent_asin).")


## 3. Read metadata and align with reviewed products

Next we read the metadata file and keep only products that appear in the review sample.
This gives us, for each `parent_asin`:

- Title
- Main category and full category path
- Average rating and rating count from metadata
- Price and store

We then join reviews and metadata into a single `joined_df` that will be used both
for EDA and for building the search index.


In [ ]:
# Read meta file and align with reviewed products

print()
print("=" * 80)
print("[STEP 2] Reading meta file and aligning with reviewed products")
print("=" * 80)

meta_records: List[Dict[str, Any]] = []
matched_count = 0
total_meta_lines = 0

parent_asins_with_reviews = set(review_count.keys())

with open(META_PATH, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        total_meta_lines += 1
        try:
            rec = json.loads(line)
        except json.JSONDecodeError:
            continue

        parent_asin = rec.get("parent_asin")
        if parent_asin is None:
            continue
        parent_asin = str(parent_asin)

        # Only keep products that have at least one review
        if parent_asin not in parent_asins_with_reviews:
            continue

        matched_count += 1

        title = rec.get("title", "")
        main_category = rec.get("main_category", "")
        categories = rec.get("categories", [])
        average_rating = rec.get("average_rating", None)
        rating_number = rec.get("rating_number", None)
        price = rec.get("price", None)
        store = rec.get("store", "")

        # Normalize categories to a compact string representation
        if isinstance(categories, list):
            categories_str = " > ".join([str(c) for c in categories])
        else:
            categories_str = str(categories)

        meta_records.append(
            {
                "parent_asin": parent_asin,
                "meta_title": title,
                "meta_main_category": main_category,
                "meta_categories": categories_str,
                "meta_average_rating": average_rating,
                "meta_rating_number": rating_number,
                "meta_price": price,
                "meta_store": store,
            }
        )

        if matched_count % 50_000 == 0:
            print(f"Matched {matched_count} products with reviews & meta...")

print(f"Total meta records scanned: {total_meta_lines}")
print(f"Total products with both meta and reviews: {matched_count}")

meta_df = pd.DataFrame(meta_records)

# Join reviews + meta
joined_df = pd.merge(reviews_agg_df, meta_df, on="parent_asin", how="inner")
print("Joined DataFrame shape:", joined_df.shape)


## 4. EDA: understanding reviews and metadata

Here we explore:

1. How many reviews each product has  
2. How long review texts are on average  
3. How ratings are distributed  
4. How review volume, average rating, and helpful votes correlate  

These insights directly motivate design choices for summarization and indexing.


In [ ]:
# EDA

print()
print("=" * 80)
print("[EDA] Exploring review and metadata characteristics")
print("=" * 80)

# 1) Review count distribution
print("\n[EDA] Review count (n_reviews) per product:")
print(joined_df["n_reviews"].describe())
pct_leq_2 = (joined_df["n_reviews"] <= 2).mean() * 100
pct_geq_20 = (joined_df["n_reviews"] >= 20).mean() * 100
print(f"Products with <= 2 reviews: {pct_leq_2:.2f}%")
print(f"Products with >= 20 reviews: {pct_geq_20:.2f}%")

# 2) Average review text length
print("\n[EDA] Average review text length per product (avg_text_len, characters):")
print(joined_df["avg_text_len"].describe())

# 3) Helpful votes
print("\n[EDA] Total helpful votes per product (total_helpful_vote):")
print(joined_df["total_helpful_vote"].describe())

# 4) Meta average rating
print("\n[EDA] Average rating from meta (meta_average_rating):")
print(joined_df["meta_average_rating"].describe())

# 5) Correlation matrix
print("\n[EDA] Correlation between n_reviews, avg_rating_reviews, total_helpful_vote:")
print(
    joined_df[["n_reviews", "avg_rating_reviews", "total_helpful_vote"]].corr()
)

# Plots
# Histogram of number of reviews (capped for visualization)
plt.figure(figsize=(6, 4))
joined_df["n_reviews_capped"] = joined_df["n_reviews"].clip(upper=50)
plt.hist(joined_df["n_reviews_capped"], bins=50)
plt.title("Number of Reviews per Product (capped at 50)")
plt.xlabel("n_reviews (capped at 50)")
plt.ylabel("Number of Products")
plt.tight_layout()
plt.show()

# Histogram of average review text length (capped for visualization)
plt.figure(figsize=(6, 4))
avg_len_capped = joined_df["avg_text_len"].clip(upper=2000)
plt.hist(avg_len_capped, bins=50)
plt.title("Average Review Text Length per Product")
plt.xlabel("Characters (capped at 2,000)")
plt.ylabel("Number of Products")
plt.tight_layout()
plt.show()

# Histogram of meta average rating
plt.figure(figsize=(6, 4))
plt.hist(joined_df["meta_average_rating"].dropna(), bins=40)
plt.title("Distribution of Meta Average Rating")
plt.xlabel("Average Rating")
plt.ylabel("Number of Products")
plt.tight_layout()
plt.show()


### 4.1 EDA interpretation and implications for our design

From the EDA above we observe:

- **Review volume is extremely skewed**  
  ~78% of products have ≤ 2 reviews, and only ~1.6% have ≥ 20 reviews.  
  ⇒ For many products we simply do not have enough textual evidence to build a
  reliable summary. This motivates adding thresholds on `n_reviews` when
  generating summaries.

- **Review texts are relatively short but with a long tail**  
  Median average length is around 140 characters, but the tail goes up to
  12,000+ characters.  
  ⇒ We want summarization to weight the most informative sentences, but also
  cap how much text we process for performance reasons.

- **Ratings are highly concentrated between 4.0 and 4.8**  
  This confirms the typical Amazon “high rating” bias and suggests that ratings
  alone are not discriminative enough; we need text to explain *why* users like
  or dislike a product.

- **Helpful votes correlate strongly with number of reviews**  
  (correlation ≈ 0.8).  
  ⇒ Helpful vote count is mostly a proxy for popularity and volume. We still
  keep it as a useful indicator but do not use it directly in ranking.

These findings lead to two concrete design changes in this version:

1. **Summarization safeguards**  
   We only generate review-based summaries when a product has at least
   `MIN_REVIEWS_FOR_SUMMARY` reviews *and* an average text length above
   `MIN_AVG_TEXT_LEN_FOR_SUMMARY`. Otherwise, we fall back to a simple, safe
   surrogate summary (the product title).

2. **Combined text construction**  
   For the search index we explicitly combine three sources: title, categories,
   and a small set of representative review snippets, instead of using reviews
   alone. This improves robustness when review coverage is sparse.


In [ ]:
# Build combined_text and fit TF-IDF

print()
print("=" * 80)
print("[STEP 3] Building combined_text for TF-IDF")
print("=" * 80)


def build_combined_text(row: pd.Series) -> str:
    """
    Build a combined text field from:
    - Product title (strong, clean signal)
    - Category path (adds topical context)
    - Up to a few cleaned review texts (user voice)
    """
    parts: List[str] = []

    title = row.get("meta_title", "")
    categories = row.get("meta_categories", "")

    if isinstance(title, str) and title.strip():
        # Repeat title once to give it slightly higher weight
        parts.append(title.strip())
        parts.append(title.strip())
    if isinstance(categories, str) and categories.strip():
        parts.append(categories.strip())

    # Add up to 5 review texts for richer semantics
    pa = str(row["parent_asin"])
    texts = reviews_sample.get(pa, [])
    if texts:
        parts.append(" ".join(texts[:5]))

    return " . ".join(parts)


joined_df["combined_text"] = joined_df.apply(build_combined_text, axis=1)
joined_df = joined_df[joined_df["combined_text"].astype(str).str.strip() != ""].reset_index(drop=True)

print("Number of products after dropping empty combined_text:", len(joined_df))

print()
print("=" * 80)
print("[STEP 4] Fitting TF-IDF vectorizer on combined_text")
print("=" * 80)

documents = joined_df["combined_text"].astype(str).tolist()
n_docs = len(documents)
print("Number of products to index:", n_docs)

vectorizer = TfidfVectorizer(
    max_features=50_000,
    stop_words="english"
)
tfidf_matrix = vectorizer.fit_transform(documents)
print("TF-IDF matrix shape:", tfidf_matrix.shape)


## 5. Save index artifacts

We now persist the artifacts that the search backend and Flask dashboard
will load:

- `products_index.csv`: per-product meta and aggregated stats  
- `tfidf_vectorizer.joblib`: fitted TF-IDF vectorizer  
- `product_tfidf.npz`: sparse document–term matrix  
- `reviews_sample.pkl`: cleaned review snippets for summarization


In [ ]:
# Save artifacts

print()
print("=" * 80)
print("[STEP 5] Saving artifacts to office_products_index/")
print("=" * 80)

products_df_to_save = joined_df.drop(columns=["combined_text"])
products_df_to_save.to_csv(PRODUCT_CSV_PATH, index=False)
print("Saved products index CSV to:", PRODUCT_CSV_PATH)

joblib.dump(vectorizer, TFIDF_VECTORIZER_PATH)
print("Saved TF-IDF vectorizer to:", TFIDF_VECTORIZER_PATH)

sparse.save_npz(TFIDF_MATRIX_PATH, tfidf_matrix)
print("Saved TF-IDF matrix to:", TFIDF_MATRIX_PATH)

with open(REVIEWS_SAMPLE_PKL, "wb") as f:
    pickle.dump(reviews_sample, f)
print("Saved review samples to:", REVIEWS_SAMPLE_PKL)

print()
print("All artifacts saved. office_products_index is ready.")


## 6. Search backend and summarization

The `SearchEngine` class:

- Loads all artifacts from disk
- Implements `search(query, top_k)` using TF-IDF + cosine similarity
- Implements a lightweight extractive summarization over review texts

To address the professor’s comment that the summaries were unclear, we now:

- Clean HTML artifacts from review text
- Require a minimum amount of review evidence before summarizing
- Fall back to the product title when we cannot reliably summarize  
- Make the summarization logic explicit and explainable


In [ ]:
# Search backend (SearchEngine)

from typing import List, Dict, Any  # re-import for clarity


class SearchEngine:
    """
    Core search engine:
    - Loads products index, TF-IDF vectorizer, TF-IDF matrix, and review samples.
    - Provides search(query, top_k) method.
    - Provides simple extractive summarization over review texts.
    """

    def __init__(self):
        self.df: pd.DataFrame = None
        self.vectorizer: TfidfVectorizer = None
        self.tfidf_matrix: sparse.spmatrix = None
        self.reviews_sample: Dict[str, List[str]] = {}

    def load(self):
        """Load index artifacts from disk."""
        if not os.path.exists(PRODUCT_CSV_PATH):
            raise FileNotFoundError(f"products_index.csv not found at {PRODUCT_CSV_PATH}")
        self.df = pd.read_csv(PRODUCT_CSV_PATH)
        self.df["parent_asin"] = self.df["parent_asin"].astype(str)

        if not os.path.exists(TFIDF_VECTORIZER_PATH):
            raise FileNotFoundError(f"Vectorizer not found at {TFIDF_VECTORIZER_PATH}")
        self.vectorizer = joblib.load(TFIDF_VECTORIZER_PATH)

        if not os.path.exists(TFIDF_MATRIX_PATH):
            raise FileNotFoundError(f"TF-IDF matrix not found at {TFIDF_MATRIX_PATH}")
        self.tfidf_matrix = sparse.load_npz(TFIDF_MATRIX_PATH)

        if not os.path.exists(REVIEWS_SAMPLE_PKL):
            print(f"WARNING: {REVIEWS_SAMPLE_PKL} not found, summaries will be basic.")
            self.reviews_sample = {}
        else:
            with open(REVIEWS_SAMPLE_PKL, "rb") as f:
                self.reviews_sample = pickle.load(f)

        print("==============================================")
        print(f"Loaded {len(self.df)} products from: {PRODUCT_CSV_PATH}")
        print(f"TF-IDF matrix shape: {self.tfidf_matrix.shape}")
        print(f"Review samples for {len(self.reviews_sample)} products")
        print("==============================================")

    def _summarize_reviews_for_product(self, parent_asin: str) -> str:
        """
        Very lightweight extractive summarization:
        1. Check whether we have enough reviews and text to summarize.
        2. Concatenate review texts for this product.
        3. Sentence split.
        4. Fit local TF-IDF on sentences and score them by sum of weights.
        5. Pick top N sentences and keep original order.

        If we do not have enough evidence, we fall back to the product title.
        """
        # Look up basic statistics for this product
        row = self.df[self.df["parent_asin"] == parent_asin]
        if row.empty:
            return ""

        n_reviews = int(row.iloc[0]["n_reviews"])
        avg_len = float(row.iloc[0]["avg_text_len"])

        # require enough reviews and text to trust a summary
        if (n_reviews < MIN_REVIEWS_FOR_SUMMARY) or (avg_len < MIN_AVG_TEXT_LEN_FOR_SUMMARY):
            return str(row.iloc[0]["meta_title"])

        texts = self.reviews_sample.get(parent_asin, [])
        if not texts:
            return str(row.iloc[0]["meta_title"])

        full_text = " ".join(texts)
        sentences = split_into_sentences(full_text)
        sentences = [s.strip() for s in sentences if len(s.strip()) > 30]

        if not sentences:
            return str(row.iloc[0]["meta_title"])

        if len(sentences) > MAX_SENTENCES_CONSIDERED:
            sentences = sentences[:MAX_SENTENCES_CONSIDERED]

        local_vectorizer = TfidfVectorizer(
            max_features=1000,
            stop_words="english"
        )
        try:
            X = local_vectorizer.fit_transform(sentences)
        except ValueError:
            # Edge cases where sentences are not diverse enough
            return " ".join(sentences[:MAX_SENTENCES_IN_SUMMARY])

        scores = np.asarray(X.sum(axis=1)).ravel()
        top_n = min(MAX_SENTENCES_IN_SUMMARY, len(sentences))
        top_idx = np.argsort(scores)[::-1][:top_n]
        selected = sorted(top_idx.tolist())
        summary_sentences = [sentences[i] for i in selected]
        return " ".join(summary_sentences)

    def search(self, query: str, top_k: int = 10) -> List[Dict[str, Any]]:
        """Return top_k products for a query with meta info and review summary."""
        query = (query or "").strip()
        if not query:
            return []

        q_vec = self.vectorizer.transform([query])
        q_vec = normalize(q_vec, norm="l2", axis=1)
        scores = q_vec.dot(self.tfidf_matrix.T).toarray()[0]

        top_k = min(top_k, len(scores))
        top_idx = np.argsort(scores)[::-1][:top_k]

        results: List[Dict[str, Any]] = []
        for idx in top_idx:
            row = self.df.iloc[idx]
            parent_asin = str(row["parent_asin"])
            item = {
                "score": float(scores[idx]),
                "parent_asin": parent_asin,
                "title": row.get("meta_title"),
                "main_category": row.get("meta_main_category"),
                "categories": row.get("meta_categories"),
                "price": row.get("meta_price"),
                "avg_rating": row.get("meta_average_rating"),
                "rating_count": row.get("meta_rating_number"),
                "n_reviews_sampled": int(row.get("n_reviews") or 0),
                "avg_text_len": float(row.get("avg_text_len") or 0.0),
                "total_helpful_vote": int(row.get("total_helpful_vote") or 0),
            }
            item["summary"] = self._summarize_reviews_for_product(parent_asin)
            results.append(item)

        return results


# Initialize global engine used by notebook and Flask
engine = SearchEngine()
engine.load()


### 6.1 Inspecting a single product summary step-by-step

To make the summarization clearer, we can inspect one product:

- Show its raw review snippets
- Show the generated summary

This makes it easy to put before/after examples on the slides.


In [ ]:
# Pick an example product (first result of a notebook query)
example_query = "a5 dotted notebook for bullet journaling"
print("=" * 80)
print(f"Test search for query: {example_query!r}")
print("=" * 80)

example_results = engine.search(example_query, top_k=1)
if example_results:
    ex = example_results[0]
    print("Title:", ex["title"])
    print("n_reviews:", ex["n_reviews_sampled"], "| avg_text_len:", ex["avg_text_len"])
    print("\nGenerated summary:\n", ex["summary"])

    # Show raw review snippets used
    pa = ex["parent_asin"]
    raw_snippets = engine.reviews_sample.get(pa, [])
    print("\nFirst 2 raw review snippets used for this summary:")
    for i, t in enumerate(raw_snippets[:2], start=1):
        print(f"[Snippet {i}] {t[:300]} ...")
else:
    print("No result found for the example query.")


## 7. Flask dashboard 

We keep the same overall look, but make two small improvements
for clarity:

- Show how many reviews the summary is based on  
- Show when the summary is actually a fallback to the product title  
  (for products with too few or too short reviews)

Note: Please stop the Flask app to continue with the evaluation.


In [ ]:
# Flask frontend


from flask import Flask, request, render_template_string

app = Flask(__name__)

HTML_TEMPLATE = """
<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <title>Office Products Search Demo</title>
  <style>
    body {
      font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
      background-color: #f5f5f5;
      margin: 0;
      padding: 0;
    }
    .container {
      max-width: 960px;
      margin: 0 auto;
      padding: 24px 16px 40px;
    }
    h1 {
      margin-top: 0;
      font-size: 28px;
    }
    .search-box {
      background: white;
      padding: 16px;
      border-radius: 8px;
      box-shadow: 0 2px 4px rgba(0,0,0,0.08);
      margin-bottom: 24px;
    }
    .search-input {
      width: 100%;
      padding: 10px 12px;
      border-radius: 4px;
      border: 1px solid #ddd;
      font-size: 16px;
      box-sizing: border-box;
    }
    .search-button {
      margin-top: 10px;
      padding: 8px 16px;
      border: none;
      border-radius: 4px;
      background-color: #2563eb;
      color: white;
      font-size: 14px;
      cursor: pointer;
    }
    .search-button:hover {
      background-color: #1d4ed8;
    }
    .result-card {
      background: white;
      padding: 16px;
      border-radius: 8px;
      box-shadow: 0 1px 3px rgba(0,0,0,0.05);
      margin-bottom: 16px;
    }
    .result-title {
      font-size: 18px;
      margin: 0 0 4px;
    }
    .result-meta {
      color: #666;
      font-size: 13px;
      margin-bottom: 8px;
    }
    .badge {
      display: inline-block;
      padding: 2px 6px;
      border-radius: 999px;
      font-size: 11px;
      background-color: #e5e7eb;
      margin-right: 4px;
    }
    .summary-source {
      font-size: 12px;
      color: #555;
      margin-top: 4px;
    }
    .result-summary-title {
      font-weight: 600;
      font-size: 14px;
      margin-bottom: 4px;
    }
    .result-summary {
      font-size: 14px;
      line-height: 1.4;
    }
    .no-results {
      margin-top: 16px;
      color: #777;
    }
    .footer {
      margin-top: 24px;
      font-size: 12px;
      color: #999;
      text-align: center;
    }
  </style>
</head>
<body>
<div class="container">
  <h1>Office Products Search + Review Summary</h1>
  <div class="search-box">
    <form method="get" action="/search">
      <input class="search-input"
             type="text"
             name="q"
             placeholder="Describe what you need (e.g. &quot;a5 dotted notebook for bullet journaling&quot;)"
             value="{{ query | default('') }}">
      <button class="search-button" type="submit">Search</button>
    </form>
  </div>

  {% if results is none %}
    <p>Try some queries, for example:</p>
    <ul>
      <li>A5 dotted notebook for bullet journaling</li>
      <li>blue waterproof fountain pen ink</li>
      <li>comfortable ergonomic office chair with lumbar support</li>
    </ul>
  {% else %}
    {% if results %}
      <p>Top {{ results|length }} search results:</p>
      {% for r in results %}
        <div class="result-card">
          <h2 class="result-title">{{ r.title }}</h2>
          <div class="result-meta">
            <span class="badge">Score {{ "%.3f"|format(r.score) }}</span>
            {% if r.avg_rating %}
              <span class="badge">{{ "%.1f"|format(r.avg_rating) }} ★</span>
            {% endif %}
            {% if r.rating_count %}
              <span class="badge">{{ r.rating_count }} ratings</span>
            {% endif %}
            {% if r.price %}
              <span class="badge">${{ r.price }}</span>
            {% endif %}
            {% if r.main_category %}
              <span class="badge">{{ r.main_category }}</span>
            {% endif %}
            <span class="badge">{{ r.n_reviews_sampled }} reviews</span>
          </div>

          <div class="result-summary-title">Review summary:</div>
          <div class="result-summary">
            {% if r.summary %}
              {{ r.summary }}
            {% else %}
              <em>No summary available.</em>
            {% endif %}
          </div>
          <div class="summary-source">
            {% if r.n_reviews_sampled >= """ + str(MIN_REVIEWS_FOR_SUMMARY) + """ and r.avg_text_len >= """ + str(MIN_AVG_TEXT_LEN_FOR_SUMMARY) + """ %}
              Summary based on review texts.
            {% else %}
              Summary falls back to product title due to limited review data.
            {% endif %}
          </div>
        </div>
      {% endfor %}
    {% else %}
      <p class="no-results">No results found. Try a different query.</p>
    {% endif %}
  {% endif %}

  <div class="footer">
    Demo for NLP Term Project — Office Products Search & Summarization
  </div>
</div>
</body>
</html>
"""


@app.route("/")
def index():
    query = request.args.get("q", "").strip()
    return render_template_string(HTML_TEMPLATE, query=query, results=None)


@app.route("/search")
def search():
    query = request.args.get("q", "").strip()
    if not query:
        return render_template_string(HTML_TEMPLATE, query="", results=[])
    results = engine.search(query, top_k=10)
    return render_template_string(HTML_TEMPLATE, query=query, results=results)
## Run App
if __name__ == "__main__":
    app.run(
        host="127.0.0.1",
        port=5000,
        debug=False,
        use_reloader=False
    )




## 8. Retrieval evaluation: Hit@k, MRR@k, nDCG@k

Finally, we evaluate the search index with a self-retrieval test:

- Randomly sample products from the index  
- Use each product title as a query  
- Treat the product itself as the only relevant item  
- Measure:
  - **Hit@k**: how often the correct product appears in the top-k
  - **MRR@k**: how early the correct product appears
  - **nDCG@k**: discounting by rank position

This is a standard information-retrieval sanity check and ties directly to the
evaluation slides.


In [ ]:
# Search evaluation

import random


def evaluate_search_engine(
    engine,
    n_queries: int = 200,
    k: int = 10,
    random_seed: int = 42,
) -> Dict[str, Any]:
    """
    Evaluate the search engine using self-retrieval:
    - Randomly sample products from engine.df
    - Use each product's title as the query
    - Treat that product's parent_asin as the only relevant item
    - Compute Hit@k, MRR@k, nDCG@k and rank distribution.
    """
    if engine.df is None or len(engine.df) == 0:
        raise ValueError("engine.df is empty. Make sure engine.load() has been called.")

    rng = random.Random(random_seed)
    all_indices = list(range(len(engine.df)))
    if n_queries > len(all_indices):
        n_queries = len(all_indices)

    sampled_indices = rng.sample(all_indices, n_queries)

    hits_at_k = 0
    mrr_sum = 0.0
    ndcg_sum = 0.0

    rank_counts: Dict[int, int] = defaultdict(int)
    not_found_count = 0

    for idx in sampled_indices:
        row = engine.df.iloc[idx]
        query = str(row.get("meta_title", "")).strip()
        parent_asin = str(row.get("parent_asin"))

        if not query:
            continue

        results = engine.search(query, top_k=k)
        rank = None
        for r_idx, r in enumerate(results):
            if str(r.get("parent_asin")) == parent_asin:
                rank = r_idx + 1
                break

        if rank is not None:
            hits_at_k += 1
            mrr_sum += 1.0 / rank
            ndcg_sum += 1.0 / np.log2(rank + 1)
            rank_counts[rank] += 1
        else:
            not_found_count += 1

    n_eval = len(sampled_indices)
    hit_at_k = hits_at_k / n_eval if n_eval > 0 else 0.0
    mrr_at_k = mrr_sum / n_eval if n_eval > 0 else 0.0
    ndcg_at_k = ndcg_sum / n_eval if n_eval > 0 else 0.0

    metrics = {
        "n_queries": n_eval,
        "k": k,
        "hit_at_k": hit_at_k,
        "mrr_at_k": mrr_at_k,
        "ndcg_at_k": ndcg_at_k,
        "rank_counts": dict(rank_counts),
        "not_found": not_found_count,
    }
    return metrics


eval_k = 10
eval_n_queries = 200

print("============================================================")
print(f"Running search evaluation: n_queries={eval_n_queries}, k={eval_k}")
print("============================================================")

metrics = evaluate_search_engine(
    engine,
    n_queries=eval_n_queries,
    k=eval_k,
    random_seed=42,
)

print("\nSummary metrics:")
print(f"  Number of evaluation queries : {metrics['n_queries']}")
print(f"  k                            : {metrics['k']}")
print(f"  Hit@{eval_k:<2d}                    : {metrics['hit_at_k']:.4f}")
print(f"  MRR@{eval_k:<2d}                    : {metrics['mrr_at_k']:.4f}")
print(f"  nDCG@{eval_k:<2d}                  : {metrics['ndcg_at_k']:.4f}")
print(f"  Not found within top-{eval_k:<2d}    : {metrics['not_found']}")

print("\nRank distribution (within top-k):")
for rank in sorted(metrics["rank_counts"].keys()):
    count = metrics["rank_counts"][rank]
    print(f"  Rank {rank:2d}: {count} queries")


# Full Hit@K curve

ks_for_curve = list(range(1, eval_k + 1))
n_eval = metrics["n_queries"]
rank_counts = metrics["rank_counts"]

hit_values = []
for k_ in ks_for_curve:
    hits = sum(count for r, count in rank_counts.items() if r <= k_)
    hit_at_k_curve = hits / n_eval if n_eval > 0 else 0.0
    hit_values.append(hit_at_k_curve)

plt.figure(figsize=(8, 4))
plt.plot(ks_for_curve, hit_values, marker="o")
plt.fill_between(ks_for_curve, hit_values, alpha=0.15)
plt.title("Hit@K Curve", fontsize=16, fontweight="bold")
plt.xlabel("K")
plt.ylabel("Hit@K")
plt.ylim(0.75, 1.02)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 8.1 Future work: embedding-based vector index

In this project we deliberately stay with a classic TF-IDF + cosine approach
because it is:

- Lightweight and deterministic
- Easy to explain and debug
- Sufficiently strong for short queries against product titles

In future iterations, we could replace or augment TF-IDF with sentence-level
embeddings, for example:

- Sentence-BERT / MiniLM encoder to get dense vectors for `combined_text`
- Approximate nearest-neighbor search (e.g., FAISS) for fast retrieval
- Hybrid scoring: combine TF-IDF score and embedding cosine similarity

This would likely improve semantic matching for queries that use different words
from the product titles, at the cost of higher model complexity and serving
infrastructure.
